# GraphInstruct-Permuted Dataset Verification

This notebook verifies the augmented dataset by showing side-by-side comparisons of original samples and their permuted versions across all 9 tasks.

In [1]:
import json
from collections import defaultdict
from IPython.display import display, Markdown, HTML

# Load the augmented dataset
samples = []
with open("GraphInstruct-Permuted/train.jsonl") as f:
    for line in f:
        samples.append(json.loads(line))

print(f"Total samples: {len(samples):,}")

# Group by original_index
groups = defaultdict(list)
for s in samples:
    groups[s["original_index"]].append(s)

print(f"Unique original samples: {len(groups):,}")
print(f"Samples per group: {len(next(iter(groups.values())))}")

# Count per task
from collections import Counter
task_counts = Counter(s["task"] for s in samples)
print("\nPer-task counts:")
for t, c in sorted(task_counts.items()):
    print(f"  {t:20s} {c:>7,}")

Total samples: 90,625
Unique original samples: 18,125
Samples per group: 5

Per-task counts:
  bipartite             10,065
  connectivity          13,435
  cycle                 13,585
  flow                   2,025
  hamilton              10,485
  shortest               6,960
  substructure          15,930
  topology               4,360
  triangle              13,780


In [2]:
def show_comparison(orig, perm, title=""):
    """Display an original sample and one permuted version side-by-side."""
    # Extract the Q: portion for cleaner display
    def q_part(query):
        idx = query.find("Q:")
        return query[idx:] if idx >= 0 else query

    orig_q = q_part(orig["query"])
    perm_q = q_part(perm["query"])

    md = f"""### {title}

**Task:** `{orig['task']}`  |  **Original index:** {orig['original_index']}  |  **Augmentation:** `{perm['augmentation']}`

---

#### Original Query
```
{orig_q}
```

#### Permuted Query
```
{perm_q}
```

#### Original Answer (last 300 chars)
```
...{orig['answer'][-300:]}
```

#### Permuted Answer (last 300 chars)
```
...{perm['answer'][-300:]}
```

---
"""
    display(Markdown(md))

## 1. One example per task: Original vs Permuted

For each of the 9 tasks, find the first original sample and show it alongside its first permuted version.

In [3]:
TASK_ORDER = [
    "connectivity", "cycle", "shortest", "bipartite", "flow",
    "topology", "hamilton", "triangle", "substructure",
]

# Find the first group for each task
task_examples = {}
for idx in sorted(groups):
    grp = groups[idx]
    task = grp[0]["task"]
    if task == "triplet":
        task = "triangle"
    if task not in task_examples:
        task_examples[task] = grp
    if len(task_examples) == len(TASK_ORDER):
        break

for task in TASK_ORDER:
    grp = task_examples[task]
    orig = grp[0]   # augmentation="original"
    perm = grp[1]   # augmentation="perm_0"
    show_comparison(orig, perm, title=f"Task: {task}")

### Task: connectivity

**Task:** `connectivity`  |  **Original index:** 0  |  **Augmentation:** `perm_0`

---

#### Original Query
```
Q: The nodes are numbered from 0 to 19, and the edges are: (0, 11) (0, 16) (0, 7) (0, 5) (0, 12) (0, 15) (0, 17) (0, 19) (0, 4) (1, 2) (1, 13) (1, 12) (1, 15) (1, 5) (1, 19) (1, 18) (1, 10) (1, 8) (2, 8) (2, 6) (2, 4) (2, 15) (2, 11) (2, 5) (2, 10) (2, 7) (2, 12) (2, 13) (3, 10) (3, 18) (3, 16) (3, 12) (3, 19) (3, 14) (3, 5) (3, 17) (3, 11) (3, 4) (4, 12) (4, 10) (4, 8) (4, 17) (4, 11) (4, 13) (4, 5) (4, 7) (4, 19) (4, 6) (4, 16) (5, 16) (5, 7) (5, 12) (5, 15) (5, 10) (5, 13) (5, 11) (6, 12) (6, 10) (6, 18) (6, 16) (6, 9) (6, 14) (6, 19) (6, 13) (7, 12) (7, 14) (7, 16) (7, 15) (7, 8) (7, 13) (8, 13) (8, 18) (8, 11) (8, 14) (8, 19) (8, 15) (9, 18) (9, 10) (9, 13) (9, 17) (9, 15) (9, 11) (10, 19) (10, 14) (10, 15) (10, 13) (10, 12) (11, 14) (11, 18) (11, 12) (11, 17) (11, 13) (12, 19) (12, 15) (12, 16) (13, 15) (13, 14) (14, 15) (15, 18) (15, 17) (15, 19) (16, 18) (17, 19) (17, 18) (18, 19). [GRAPH_REPR] Is there a path between node 6 and node 9?
```

#### Permuted Query
```
Q: The nodes are numbered from 0 to 19, and the edges are: (5,14) (1,16) (8,0) (12,11) (13,7) (5,17) (15,3) (6,10) (19,13) (7,0) (9,15) (14,16) (4,7) (10,2) (4,3) (13,18) (15,11) (4,1) (2,16) (14,6) (11,16) (19,18) (18,7) (12,10) (4,10) (17,1) (19,3) (19,7) (10,0) (12,16) (14,1) (1,3) (15,12) (6,3) (11,2) (15,7) (9,17) (6,2) (4,13) (13,1) (15,2) (9,1) (19,1) (9,8) (17,11) (12,8) (15,1) (9,10) (17,16) (14,10) (13,16) (0,3) (14,13) (16,0) (9,6) (18,1) (5,6) (18,16) (19,8) (13,10) (5,3) (10,8) (18,2) (16,8) (4,0) (12,0) (19,9) (14,17) (9,3) (4,9) (6,11) (5,11) (17,3) (5,0) (18,6) (15,0) (19,16) (4,2) (12,17) (9,13) (9,18) (5,16) (9,7) (5,1) (5,13) (8,3) (14,11) (14,15) (1,7) (9,11) (13,11) (16,3) (14,9) (19,10) (4,8) (6,16) (15,17) (4,17) (13,17) (18,11) (10,1) (17,2) (10,11) (14,18) (6,0). [GRAPH_REPR] Is there a path between node 15 and node 12?
```

#### Original Answer (last 300 chars)
```
... Node 6 and node 9 are connected by the edge (6,9), so there is a path between them. ### Yes.
```

#### Permuted Answer (last 300 chars)
```
... Node 15 and node 12 are connected by the edge (15,12), so there is a path between them. ### Yes.
```

---


### Task: cycle

**Task:** `cycle`  |  **Original index:** 2687  |  **Augmentation:** `perm_0`

---

#### Original Query
```
Q: The nodes are numbered from 0 to 9, and the edges are: (0, 9) (0, 6) (0, 1) (0, 5) (0, 7) (1, 9) (1, 7) (1, 4) (1, 2) (2, 7) (2, 3) (2, 8) (2, 5) (2, 9) (3, 6) (3, 9) (3, 8) (4, 8) (4, 6) (6, 8) (6, 9). [GRAPH_REPR] Is there a cycle in this graph?
```

#### Permuted Query
```
Q: The nodes are numbered from 0 to 9, and the edges are: (1,0) (1,4) (6,0) (8,9) (0,4) (5,1) (8,7) (8,4) (8,3) (7,4) (5,2) (7,6) (5,9) (7,5) (1,2) (5,4) (8,0) (0,2) (6,2) (5,3) (7,9). [GRAPH_REPR] Is there a cycle in this graph?
```

#### Original Answer (last 300 chars)
```
...to node 9 (via edge 0-9), then to node 3 (via edge 9-3), then to node 6 (via edge 3-6), and back to node 0 (via edge 6-0). This forms a cycle [0-9-3-6-0] without revisiting any edge. There may be other cycles in this graph as well, but finding one is enough to conclude that there is a cycle.### Yes.
```

#### Permuted Answer (last 300 chars)
```
...to node 4 (via edge 0-9), then to node 1 (via edge 9-3), then to node 0 (via edge 3-6), and back to node 8 (via edge 6-0). This forms a cycle [8-4-1-0-8] without revisiting any edge. There may be other cycles in this graph as well, but finding one is enough to conclude that there is a cycle.### Yes.
```

---


### Task: shortest

**Task:** `shortest`  |  **Original index:** 12429  |  **Augmentation:** `perm_0`

---

#### Original Query
```
Q: The nodes are numbered from 0 to 37, and the edges are: (0,16,4) (0,23,3) (0,32,10) (0,27,5) (0,12,2) (0,24,9) (0,36,7) (1,13,3) (1,5,1) (1,31,4) (1,28,1) (1,2,9) (1,22,5) (1,24,1) (1,7,10) (1,6,1) (2,25,4) (2,13,4) (2,34,2) (2,5,7) (2,18,10) (2,19,8) (2,27,1) (3,35,9) (3,13,2) (3,32,1) (3,28,6) (3,17,8) (3,37,3) (4,16,5) (4,35,4) (5,31,8) (5,34,4) (5,15,5) (5,10,2) (5,20,2) (5,26,2) (6,8,3) (6,32,7) (6,12,6) (6,9,5) (7,36,10) (7,23,8) (7,24,8) (7,16,1) (7,27,7) (7,8,7) (8,12,6) (8,33,7) (8,35,4) (8,31,7) (9,37,2) (9,27,10) (9,33,7) (10,23,10) (10,27,6) (10,20,10) (10,19,8) (10,21,7) (10,30,8) (10,26,8) (10,18,9) (10,37,1) (11,25,2) (11,34,10) (11,16,2) (11,26,3) (11,12,8) (11,27,9) (11,33,4) (12,36,6) (12,25,6) (12,24,6) (12,19,10) (12,35,4) (12,15,3) (13,37,1) (13,20,4) (14,30,8) (14,34,6) (14,29,4) (14,35,9) (14,26,3) (15,19,9) (15,28,4) (15,23,5) (15,37,4) (15,17,1) (15,18,2) (15,36,4) (16,17,4) (17,33,9) (17,22,7) (18,34,5) (18,25,9) (19,30,2) (19,31,6) (19,20,4) (20,34,8) (20,37,1) (20,29,9) (20,30,4) (20,21,5) (21,26,1) (21,34,7) (21,36,8) (21,28,10) (21,35,10) (21,33,9) (22,29,10) (22,33,2) (22,35,1) (23,35,1) (23,26,2) (24,35,6) (24,31,4) (24,36,5) (25,36,4) (26,31,7) (26,32,1) (26,33,8) (26,35,1) (27,34,10) (27,30,5) (28,37,5) (28,30,8) (28,31,5) (28,29,5) (29,33,10) (29,31,3) (30,34,6) (31,37,10) (31,33,2) (32,36,1) (34,37,10). [GRAPH_REPR] Give the weight of the shortest path from node 7 to node 33.
```

#### Permuted Query
```
Q: The nodes are numbered from 0 to 37, and the edges are: (33,12,1) (32,16,10) (20,30,9) (27,26,3) (17,35,9) (12,2,8) (10,16,1) (33,6,7) (3,30,10) (20,37,6) (7,9,4) (32,37,7) (36,10,1) (13,0,4) (9,6,8) (7,14,1) (29,12,2) (31,0,2) (33,19,10) (20,9,10) (32,1,8) (29,17,5) (9,33,5) (23,16,6) (33,16,8) (25,37,10) (19,14,5) (37,5,5) (35,5,2) (36,7,2) (19,26,5) (26,2,2) (25,14,2) (13,1,9) (19,5,8) (23,1,6) (5,6,6) (8,1,1) (31,2,4) (27,2,10) (21,12,2) (4,5,8) (20,33,7) (23,17,3) (17,19,4) (4,34,9) (1,16,5) (31,6,10) (9,27,9) (1,34,6) (17,16,4) (36,34,9) (8,32,10) (30,15,9) (28,10,7) (11,26,7) (35,9,4) (18,27,10) (33,2,9) (13,37,5) (36,14,3) (31,23,8) (15,16,4) (20,5,8) (13,23,2) (3,35,8) (24,18,7) (22,34,4) (8,7,3) (20,35,8) (29,20,2) (29,6,4) (30,6,5) (24,2,9) (32,0,1) (35,26,6) (22,0,5) (1,26,4) (4,6,6) (12,10,1) (4,12,3) (12,26,7) (13,21,3) (29,26,8) (23,35,10) (28,25,5) (37,6,10) (13,10,10) (8,29,1) (17,14,4) (4,27,4) (33,34,10) (32,21,8) (36,19,6) (11,23,6) (23,15,6) (17,21,5) (0,24,4) (3,6,2) (9,5,4) (13,16,7) (36,24,8) (11,2,7) (17,30,2) (8,28,1) (11,34,4) (3,15,4) (31,37,9) (3,29,7) (8,3,9) (18,34,1) (8,26,4) (20,21,10) (25,2,7) (18,2,2) (17,24,1) (3,7,4) (3,37,1) (9,14,1) (8,19,1) (26,14,10) (12,34,1) (8,18,5) (28,11,3) (32,11,7) (28,23,6) (21,34,1) (19,27,5) (31,12,3) (31,15,2) (6,14,10) (20,14,1) (23,34,4) (20,12,8) (29,9,2). [GRAPH_REPR] Give the weight of the shortest path from node 32 to node 2.
```

#### Original Answer (last 300 chars)
```
...At this point, we have reached the destination node 33, so we can stop the algorithm and trace back the shortest path by following the previous nodes. The shortest path from node 7 to node 33 is [7, 16, 11, 33] with a total weight of 7. ### 7.
```

#### Permuted Answer (last 300 chars)
```
...At this point, we have reached the destination node 2, so we can stop the algorithm and trace back the shortest path by following the previous nodes. The shortest path from node 32 to node 2 is [32, 0, 31, 2] with a total weight of 7. ### 7.
```

---


### Task: bipartite

**Task:** `bipartite`  |  **Original index:** 5563  |  **Augmentation:** `perm_0`

---

#### Original Query
```
Q: The nodes are numbered from 0 to 1, and the edges are: (0->1). [GRAPH_REPR] Is this graph bipartite?
```

#### Permuted Query
```
Q: The nodes are numbered from 0 to 1, and the edges are: (0->1). [GRAPH_REPR] Is this graph bipartite?
```

#### Original Answer (last 300 chars)
```
...hat no two nodes in the same group have a direct edge. 
In this case, the graph has only two nodes, 0 and 1, and one edge, (0->1). 
We can easily assign node 0 to one group and node 1 to another group, and there will be no conflict. 
Therefore, the graph is bipartite, and the answer is yes. ### Yes.
```

#### Permuted Answer (last 300 chars)
```
...hat no two nodes in the same group have a direct edge. 
In this case, the graph has only two nodes, 0 and 1, and one edge, (0->1). 
We can easily assign node 0 to one group and node 1 to another group, and there will be no conflict. 
Therefore, the graph is bipartite, and the answer is yes. ### Yes.
```

---


### Task: flow

**Task:** `flow`  |  **Original index:** 5404  |  **Augmentation:** `perm_0`

---

#### Original Query
```
Q: The nodes are numbered from 0 to 7, and the edges are: (0->7,8) (0->4,1) (0->2,9) (0->6,1) (1->7,8) (1->3,10) (1->6,5) (1->5,1) (4->5,2) (5->6,6) (6->7,2). [GRAPH_REPR] What is the maximum flow from node 4 to node 7?
```

#### Permuted Query
```
Q: The nodes are numbered from 0 to 7, and the edges are: (1->6,8) (7->2,6) (3->7,2) (0->5,10) (0->7,1) (1->3,1) (1->4,9) (0->6,8) (2->6,2) (0->2,5) (1->2,1). [GRAPH_REPR] What is the maximum flow from node 3 to node 6?
```

#### Original Answer (last 300 chars)
```
...The maximum flow from node 4 to node 7 is the sum of the flows along the edges that leave node 4, which is 2 + 0 = 2 units. Alternatively, it is also the sum of the flows along the edges that enter node 7, which is also 2 units. ### 2.
```

#### Permuted Answer (last 300 chars)
```
...The maximum flow from node 3 to node 6 is the sum of the flows along the edges that leave node 3, which is 2 + 0 = 2 units. Alternatively, it is also the sum of the flows along the edges that enter node 6, which is also 2 units. ### 2.
```

---


### Task: topology

**Task:** `topology`  |  **Original index:** 13821  |  **Augmentation:** `perm_0`

---

#### Original Query
```
Q: The nodes are numbered from 0 to 16, and the edges are: (0->10) (0->1) (0->7) (0->16) (0->4) (0->3) (0->2) (0->15) (0->6) (0->13) (0->8) (0->11) (1->16) (1->4) (1->7) (1->14) (1->6) (1->5) (1->2) (1->11) (1->12) (1->15) (1->10) (1->9) (1->13) (2->15) (2->9) (2->3) (2->12) (2->14) (2->11) (2->4) (2->8) (2->6) (2->10) (2->16) (3->15) (3->14) (3->13) (3->4) (3->8) (3->9) (3->7) (3->16) (3->6) (3->12) (4->14) (4->16) (4->10) (4->9) (4->5) (4->6) (4->12) (4->11) (5->8) (5->7) (5->9) (5->14) (5->12) (5->11) (5->13) (5->6) (6->12) (6->8) (6->9) (6->13) (6->15) (6->14) (6->16) (6->11) (6->10) (7->10) (7->15) (7->12) (7->16) (7->11) (7->8) (8->14) (8->11) (8->13) (8->12) (8->15) (8->9) (8->10) (9->13) (9->12) (9->15) (10->16) (10->15) (10->13) (10->11) (11->16) (11->15) (11->14) (11->12) (12->15) (12->13) (12->16) (12->14) (13->14) (13->15) (14->15) (14->16) (15->16). [GRAPH_REPR] Give one topology sorting path of this graph.
```

#### Permuted Query
```
Q: The nodes are numbered from 0 to 16, and the edges are: (0->16) (13->4) (2->5) (0->4) (12->10) (12->6) (11->1) (6->16) (3->11) (7->13) (7->15) (1->6) (11->13) (1->10) (7->12) (0->6) (1->12) (5->15) (0->15) (0->13) (11->4) (9->13) (8->1) (3->14) (2->15) (2->6) (5->14) (0->7) (2->4) (3->10) (2->10) (2->9) (5->7) (0->9) (8->12) (11->6) (16->13) (12->9) (5->1) (8->15) (12->13) (8->0) (8->2) (10->14) (1->14) (0->14) (7->9) (3->9) (8->14) (5->4) (16->15) (9->16) (6->14) (11->9) (2->3) (3->4) (8->6) (13->15) (11->12) (1->15) (2->16) (6->15) (8->3) (7->10) (2->12) (3->6) (0->3) (0->10) (16->4) (10->6) (12->15) (5->13) (8->5) (3->7) (8->7) (0->2) (8->10) (7->6) (3->16) (10->15) (4->14) (12->16) (15->14) (9->15) (11->16) (16->14) (0->11) (5->3) (8->13) (7->14) (0->1) (7->4) (4->15) (5->12) (11->7) (7->16) (10->13) (12->4) (5->16) (2->7) (2->14) (1->16) (5->9) (6->4). [GRAPH_REPR] Give one topology sorting path of this graph.
```

#### Original Answer (last 300 chars)
```
...ges. Node 1 is the next choice because its only incoming edge is from node 0, which has been removed from the graph. Node 2 is the next choice because its incoming edges are from nodes 0 and 1, both of which have been removed, and so on. ### [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
```

#### Permuted Answer (last 300 chars)
```
...ges. Node 0 is the next choice because its only incoming edge is from node 8, which has been removed from the graph. Node 2 is the next choice because its incoming edges are from nodes 0 and 1, both of which have been removed, and so on. ### [8, 0, 2, 5, 3, 11, 7, 1, 12, 9, 10, 6, 16, 13, 4, 15, 14]
```

---


### Task: hamilton

**Task:** `hamilton`  |  **Original index:** 7576  |  **Augmentation:** `perm_0`

---

#### Original Query
```
Q: The nodes are numbered from 0 to 1, and the edges are: (0, 1). [GRAPH_REPR] Is there a Hamiltonian path in this graph?
```

#### Permuted Query
```
Q: The nodes are numbered from 0 to 1, and the edges are: (1,0). [GRAPH_REPR] Is there a Hamiltonian path in this graph?
```

#### Original Answer (last 300 chars)
```
...one edge. Therefore, any path that starts from node 0 and ends at node 1, or vice versa, will be a Hamiltonian path. 
For example, one possible Hamiltonian path is: [0, 1]. Another possible Hamiltonian path is: [1, 0]. 
Therefore, there is a Hamiltonian path in this graph. ### Yes, [0, 1] or [1, 0].
```

#### Permuted Answer (last 300 chars)
```
...one edge. Therefore, any path that starts from node 1 and ends at node 0, or vice versa, will be a Hamiltonian path. 
For example, one possible Hamiltonian path is: [1, 0]. Another possible Hamiltonian path is: [0, 1]. 
Therefore, there is a Hamiltonian path in this graph. ### Yes, [1, 0] or [0, 1].
```

---


### Task: triangle

**Task:** `triangle`  |  **Original index:** 9673  |  **Augmentation:** `perm_0`

---

#### Original Query
```
Q: The nodes are numbered from 0 to 12, weights of nodes are: [0, 9] [1, 3] [2, 6] [3, 3] [4, 6] [5, 4] [6, 7] [7, 9] [8, 4] [9, 1] [10, 7] [11, 2] [12, 9], and the edges are: (0, 10) (0, 9) (2, 6) (7, 9) (9, 10). [GRAPH_REPR] What is the maximum sum of the weights of three nodes?
```

#### Permuted Query
```
Q: The nodes are numbered from 0 to 12, weights of nodes are: [0, 9] [1, 3] [2, 9] [3, 6] [4, 9] [5, 7] [6, 4] [7, 6] [8, 7] [9, 3] [10, 2] [11, 1] [12, 4], and the edges are: (0,11) (4,11) (0,8) (3,5) (11,8). [GRAPH_REPR] What is the maximum sum of the weights of three nodes?
```

#### Original Answer (last 300 chars)
```
...is 17. This is because Node 9 is the only node that is connected to both groups, and it has a weight of 1. 
There are no other groups of three interconnected nodes in the graph, so this is the maximum possible sum. 
### The maximum sum of the weights of three interconnected nodes in the graph is 17.
```

#### Permuted Answer (last 300 chars)
```
...s 17. This is because Node 11 is the only node that is connected to both groups, and it has a weight of 1. 
There are no other groups of three interconnected nodes in the graph, so this is the maximum possible sum. 
### The maximum sum of the weights of three interconnected nodes in the graph is 17.
```

---


### Task: substructure

**Task:** `substructure`  |  **Original index:** 14693  |  **Augmentation:** `perm_0`

---

#### Original Query
```
Q: The nodes of graph G are numbered from 0 to 6, and the edges are: (0->2) (0->4) (1->3) (2->5) (3->4) (4->5). The nodes of subgraph G' are numbered from a to d, and the edges are: (a->b) (a->d) (a->c) (b->d) (b->c) (c->d). [GRAPH_REPR] Is subgraph G' present within graph G as a direct substructure?
```

#### Permuted Query
```
Q: The nodes of graph G are numbered from 0 to 6, and the edges are: (0->6) (4->3) (1->4) (1->6) (6->3) (2->0). The nodes of subgraph G' are numbered from a to d, and the edges are: (a->b) (a->d) (a->c) (b->d) (b->c) (c->d). [GRAPH_REPR] Is subgraph G' present within graph G as a direct substructure?
```

#### Original Answer (last 300 chars)
```
...Therefore, there is no node in G that matches the pattern of 'a' in G', and thus, there is no exact match for subgraph G' within graph G. ### No.
```

#### Permuted Answer (last 300 chars)
```
...Therefore, there is no node in G that matches the pattern of 'b' in G', and thus, there is no exact match for subgraph G' within graph G. ### No.
```

---


## 2. Verify augmentation correctness

Check that:
1. `[GRAPH_REPR]` token is present in every query
2. Node IDs are consistently remapped (same node count, same edge count)
3. Numeric answers are preserved for numeric tasks
4. Yes/No answers are preserved for classification tasks

In [4]:
import re

# --- Check 1: [GRAPH_REPR] present in every query ---
missing_repr = sum(1 for s in samples if "[GRAPH_REPR]" not in s["query"])
print(f"[Check 1] Samples missing [GRAPH_REPR]: {missing_repr} / {len(samples)}")

# --- Check 2: Edge count preserved ---
def count_edges(query):
    return len(re.findall(r"\([^)]+\)", query[query.find("the edges are"):] if "the edges are" in query else ""))

edge_mismatches = 0
for idx, grp in groups.items():
    orig = grp[0]
    for aug in grp[1:]:
        if count_edges(orig["query"]) != count_edges(aug["query"]):
            edge_mismatches += 1
print(f"[Check 2] Edge count mismatches: {edge_mismatches} / {sum(len(g)-1 for g in groups.values())}")

# --- Check 3: Numeric answers preserved ---
def extract_final_answer(answer):
    if "###" in answer:
        return answer.split("###")[-1].strip().rstrip(".")
    return ""

NUMERIC_TASKS = {"shortest", "flow", "triangle", "triplet", "diameter"}
YESNO_TASKS = {"connectivity", "cycle", "bipartite", "hamilton", "substructure"}

numeric_mismatches = 0
numeric_total = 0
for idx, grp in groups.items():
    orig = grp[0]
    if orig["task"] not in NUMERIC_TASKS:
        continue
    orig_ans = extract_final_answer(orig["answer"])
    for aug in grp[1:]:
        aug_ans = extract_final_answer(aug["answer"])
        numeric_total += 1
        # Extract numbers for comparison
        orig_nums = re.findall(r"[\d.]+", orig_ans)
        aug_nums = re.findall(r"[\d.]+", aug_ans)
        if orig_nums and aug_nums:
            if abs(float(orig_nums[-1]) - float(aug_nums[-1])) > 0.01:
                numeric_mismatches += 1
        elif orig_nums != aug_nums:
            numeric_mismatches += 1

print(f"[Check 3] Numeric answer mismatches: {numeric_mismatches} / {numeric_total}")

# --- Check 4: Yes/No answers preserved ---
yesno_mismatches = 0
yesno_total = 0
for idx, grp in groups.items():
    orig = grp[0]
    if orig["task"] not in YESNO_TASKS:
        continue
    orig_ans = extract_final_answer(orig["answer"]).lower()
    orig_yn = "yes" if "yes" in orig_ans else ("no" if "no" in orig_ans else "?")
    for aug in grp[1:]:
        aug_ans = extract_final_answer(aug["answer"]).lower()
        aug_yn = "yes" if "yes" in aug_ans else ("no" if "no" in aug_ans else "?")
        yesno_total += 1
        if orig_yn != aug_yn:
            yesno_mismatches += 1

print(f"[Check 4] Yes/No answer mismatches: {yesno_mismatches} / {yesno_total}")
print()
print("All checks passed!" if (missing_repr + edge_mismatches + numeric_mismatches + yesno_mismatches) == 0 else "Some checks failed - see above.")

[Check 1] Samples missing [GRAPH_REPR]: 0 / 90625


[Check 2] Edge count mismatches: 0 / 72500
[Check 3] Numeric answer mismatches: 0 / 18212
[Check 4] Yes/No answer mismatches: 0 / 50800

All checks passed!


## 3. Verify node permutation is actually applied

For a single connectivity sample, extract and compare the node IDs used in the original vs. permuted query to confirm the bijection.

In [5]:
# Pick the first connectivity group
grp = task_examples["connectivity"]
orig = grp[0]
perm = grp[1]

def extract_edge_tuples(query):
    """Extract numeric edge tuples from the query."""
    edge_region = query[query.find("the edges are:"):]
    return re.findall(r"\((\d+)[,\s]+(\d+)\)", edge_region)

def extract_query_nodes(query):
    """Extract 'node X' references after [GRAPH_REPR]."""
    after = query[query.find("[GRAPH_REPR]"):] if "[GRAPH_REPR]" in query else query
    return re.findall(r"node\s+(\d+)", after, re.IGNORECASE)

orig_edges = extract_edge_tuples(orig["query"])
perm_edges = extract_edge_tuples(perm["query"])

print(f"Original: {len(orig_edges)} edges, query nodes = {extract_query_nodes(orig['query'])}")
print(f"Permuted: {len(perm_edges)} edges, query nodes = {extract_query_nodes(perm['query'])}")
print()

# Show first 5 edges of each
print("Original edges (first 5):", orig_edges[:5])
print("Permuted edges (first 5):", perm_edges[:5])
print()

# Verify the set of node IDs is the same (just relabeled)
orig_node_set = set()
for u, v in orig_edges:
    orig_node_set.update([int(u), int(v)])

perm_node_set = set()
for u, v in perm_edges:
    perm_node_set.update([int(u), int(v)])

print(f"Original node ID set: {sorted(orig_node_set)}")
print(f"Permuted node ID set: {sorted(perm_node_set)}")
print(f"Same set of IDs? {orig_node_set == perm_node_set}")
print(f"Edge order differs?  {orig_edges != perm_edges}")

Original: 105 edges, query nodes = ['6', '9']
Permuted: 105 edges, query nodes = ['15', '12']

Original edges (first 5): [('0', '11'), ('0', '16'), ('0', '7'), ('0', '5'), ('0', '12')]
Permuted edges (first 5): [('5', '14'), ('1', '16'), ('8', '0'), ('12', '11'), ('13', '7')]

Original node ID set: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Permuted node ID set: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Same set of IDs? True
Edge order differs?  True


## 4. Verify graph isomorphism (structural equivalence)

Reconstruct the graph from edges for both original and permuted versions, and verify they are isomorphic using NetworkX.

In [6]:
import networkx as nx
import random

def build_graph(query, directed=False):
    """Build a NetworkX graph from a query string's edge list."""
    G = nx.DiGraph() if directed else nx.Graph()
    edge_region = query[query.find("the edges are:"):]
    if directed:
        for m in re.finditer(r"\((\d+)->(\d+)(?:,\s*\d+)?\)", edge_region):
            G.add_edge(int(m.group(1)), int(m.group(2)))
    else:
        for m in re.finditer(r"\((\d+)[,\s]+(\d+)(?:[,\s]+\d+)?\)", edge_region):
            G.add_edge(int(m.group(1)), int(m.group(2)))
    return G

# Test isomorphism on 20 random groups across various tasks
random.seed(42)
test_indices = random.sample(list(groups.keys()), min(20, len(groups)))

iso_results = []
for idx in test_indices:
    grp = groups[idx]
    orig = grp[0]
    perm = grp[1]
    task = orig["task"]
    directed = task in ("bipartite", "flow", "topology", "substructure")

    G_orig = build_graph(orig["query"], directed)
    G_perm = build_graph(perm["query"], directed)

    if directed:
        is_iso = nx.is_isomorphic(G_orig, G_perm)
    else:
        is_iso = nx.is_isomorphic(G_orig, G_perm)

    iso_results.append((idx, task, G_orig.number_of_nodes(), G_orig.number_of_edges(), is_iso))

print(f"{'Idx':>6}  {'Task':>15}  {'Nodes':>5}  {'Edges':>5}  {'Isomorphic':>10}")
print("-" * 50)
for idx, task, n, e, iso in iso_results:
    print(f"{idx:>6}  {task:>15}  {n:>5}  {e:>5}  {'YES' if iso else 'NO':>10}")

all_iso = all(r[4] for r in iso_results)
print(f"\nAll {len(iso_results)} tested pairs isomorphic: {all_iso}")

   Idx             Task  Nodes  Edges  Isomorphic
--------------------------------------------------
  3648            cycle      5      6         YES
   819     connectivity     14     21         YES
  9012         hamilton     16     90         YES
  8024         hamilton     10     33         YES
  7314        bipartite     76    189         YES
  4572            cycle     30     71         YES
  3358            cycle     12     26         YES
 17870     substructure     23     37         YES
  2848            cycle     14     51         YES
 13825         topology     17    103         YES
  1041     connectivity      6      7         YES
   976     connectivity     18     69         YES
  3070            cycle     12     28         YES
  7164        bipartite     46    109         YES
  7623         hamilton      5      7         YES
 16559     substructure      8      6         YES
   869     connectivity      8     13         YES
  6515        bipartite      7      6         YES

## 5. Show all 5 versions of a single sample

Display the original + 4 permutations for one connectivity sample to see the full augmentation fan-out.

In [7]:
# Pick a small connectivity sample for readability
grp = task_examples["connectivity"]

for i, s in enumerate(grp):
    q = s["query"]
    q_part = q[q.find("Q:"):]
    a = s["answer"]

    label = s["augmentation"]
    print(f"===== [{label}] =====")
    print(f"Query:  {q_part[:300]}{'...' if len(q_part)>300 else ''}")
    print(f"Answer: {a[:200]}{'...' if len(a)>200 else ''}")
    print()

===== [original] =====
Query:  Q: The nodes are numbered from 0 to 19, and the edges are: (0, 11) (0, 16) (0, 7) (0, 5) (0, 12) (0, 15) (0, 17) (0, 19) (0, 4) (1, 2) (1, 13) (1, 12) (1, 15) (1, 5) (1, 19) (1, 18) (1, 10) (1, 8) (2, 8) (2, 6) (2, 4) (2, 15) (2, 11) (2, 5) (2, 10) (2, 7) (2, 12) (2, 13) (3, 10) (3, 18) (3, 16) (3, ...
Answer:  Node 6 and node 9 are connected by the edge (6,9), so there is a path between them. ### Yes.

===== [perm_0] =====
Query:  Q: The nodes are numbered from 0 to 19, and the edges are: (5,14) (1,16) (8,0) (12,11) (13,7) (5,17) (15,3) (6,10) (19,13) (7,0) (9,15) (14,16) (4,7) (10,2) (4,3) (13,18) (15,11) (4,1) (2,16) (14,6) (11,16) (19,18) (18,7) (12,10) (4,10) (17,1) (19,3) (19,7) (10,0) (12,16) (14,1) (1,3) (15,12) (6,3) ...
Answer:  Node 15 and node 12 are connected by the edge (15,12), so there is a path between them. ### Yes.

===== [perm_1] =====
Query:  Q: The nodes are numbered from 0 to 19, and the edges are: (0,18) (11,10) (5,12) (0,5) (8,15)

---

# GraphInstruct-Aug Verification

Verify the final combined training set (`GraphInstruct-Aug/train.jsonl`) that merges the original GraphInstruct (with `[GRAPH_REPR]`) and GraphInstruct-Permuted.

## 6. Load GraphInstruct-Aug and check basic statistics

In [8]:
aug_samples = []
with open("GraphInstruct-Aug/train.jsonl") as f:
    for line in f:
        aug_samples.append(json.loads(line))

print(f"Total samples in GraphInstruct-Aug: {len(aug_samples):,}")

aug_type_counts = Counter(s["augmentation"] for s in aug_samples)
print(f"\nBy augmentation type:")
for t, c in sorted(aug_type_counts.items()):
    print(f"  {t:20s} {c:>7,}")

aug_task_counts = Counter(s["task"] for s in aug_samples)
print(f"\nBy task:")
for t, c in sorted(aug_task_counts.items()):
    print(f"  {t:20s} {c:>7,}")

n_orig = aug_type_counts.get("none", 0)
n_permuted_orig = aug_type_counts.get("original", 0)
n_perm = sum(c for t, c in aug_type_counts.items() if t.startswith("perm_"))
print(f"\nBreakdown:")
print(f"  Original (augmentation='none'):      {n_orig:>7,}")
print(f"  Permuted-original (with GRAPH_REPR): {n_permuted_orig:>7,}")
print(f"  Permuted copies (perm_0..perm_3):    {n_perm:>7,}")
print(f"  Total:                               {n_orig + n_permuted_orig + n_perm:>7,}")

Total samples in GraphInstruct-Aug: 108,750

By augmentation type:
  none                  18,125
  original              18,125
  perm_0                18,125
  perm_1                18,125
  perm_2                18,125
  perm_3                18,125

By task:
  bipartite             12,078
  connectivity          16,122
  cycle                 16,302
  flow                   2,430
  hamilton              12,582
  shortest               8,352
  substructure          19,116
  topology               5,232
  triangle              16,536

Breakdown:
  Original (augmentation='none'):       18,125
  Permuted-original (with GRAPH_REPR):  18,125
  Permuted copies (perm_0..perm_3):     72,500
  Total:                               108,750


## 7. Verify all samples have `[GRAPH_REPR]` and valid fields

In [9]:
# Check 1: [GRAPH_REPR] in every query
missing_repr = sum(1 for s in aug_samples if "[GRAPH_REPR]" not in s["query"])
print(f"[Check 1] [GRAPH_REPR] missing: {missing_repr} / {len(aug_samples)}")

# Check 2: Every sample has required fields
required_fields = {"query", "answer", "task", "augmentation", "original_index"}
missing_fields = sum(1 for s in aug_samples if not required_fields.issubset(s.keys()))
print(f"[Check 2] Missing required fields: {missing_fields} / {len(aug_samples)}")

# Check 3: Every query contains 'the edges are:'
missing_edges = sum(1 for s in aug_samples if "the edges are:" not in s["query"])
print(f"[Check 3] Missing edge list: {missing_edges} / {len(aug_samples)}")

# Check 4: Every answer contains '###'
missing_hash = [i for i, s in enumerate(aug_samples) if "###" not in s["answer"]]
print(f"[Check 4] Missing ### in answer: {len(missing_hash)} / {len(aug_samples)}")
if missing_hash:
    n_source = len(set(aug_samples[i]["original_index"] for i in missing_hash))
    print(f"          -> {n_source} unique source samples x 6 copies = {n_source * 6}")
    print(f"          (pre-existing in original GraphInstruct, {n_source/18125*100:.1f}% of source)")

# Check 5: No empty queries or answers
empty_q = sum(1 for s in aug_samples if len(s["query"].strip()) == 0)
empty_a = sum(1 for s in aug_samples if len(s["answer"].strip()) == 0)
print(f"[Check 5] Empty queries: {empty_q}, Empty answers: {empty_a}")

critical = missing_repr + missing_fields + missing_edges + empty_q + empty_a
print()
print("All critical checks passed!" if critical == 0 else "Critical checks FAILED!")
if missing_hash:
    print(f"Note: {len(missing_hash)} samples lack ### (inherited from source data, not a pipeline bug).")

[Check 1] [GRAPH_REPR] missing: 0 / 108750
[Check 2] Missing required fields: 0 / 108750
[Check 3] Missing edge list: 0 / 108750
[Check 4] Missing ### in answer: 504 / 108750
          -> 84 unique source samples x 6 copies = 504
          (pre-existing in original GraphInstruct, 0.5% of source)
[Check 5] Empty queries: 0, Empty answers: 0

All critical checks passed!
Note: 504 samples lack ### (inherited from source data, not a pipeline bug).


## 8. Compare the three versions of the same sample

For each task, show the same graph instance across:
- **`none`** — original GraphInstruct with `[GRAPH_REPR]` inserted (no permutation)
- **`original`** — from GraphInstruct-Permuted (identical content, stored in the permuted file)
- **`perm_0`** — first permuted copy

In [10]:
aug_groups = defaultdict(list)
for s in aug_samples:
    aug_groups[s["original_index"]].append(s)

# Find one example per task
task_aug_examples = {}
for idx in sorted(aug_groups):
    grp = aug_groups[idx]
    task = grp[0]["task"]
    if task not in task_aug_examples:
        task_aug_examples[task] = grp
    if len(task_aug_examples) >= len(TASK_ORDER):
        break

def q_part(query):
    idx = query.find("Q:")
    return query[idx:] if idx >= 0 else query

for task in TASK_ORDER:
    grp = task_aug_examples[task]
    by_aug = {s["augmentation"]: s for s in grp}

    md = f"### {task}\n\n"
    for aug_key in ["none", "original", "perm_0"]:
        if aug_key not in by_aug:
            continue
        s = by_aug[aug_key]
        q = q_part(s["query"])
        q_display = q[:350] + ("..." if len(q) > 350 else "")
        a_final = s["answer"].split("###")[-1].strip() if "###" in s["answer"] else s["answer"][-100:]
        md += f"**`{aug_key}`**\n```\n{q_display}\n```\n**Answer** -> `### {a_final}`\n\n"

    display(Markdown(md))

### connectivity

**`none`**
```
Q: The nodes are numbered from 0 to 19, and the edges are: (0, 11) (0, 16) (0, 7) (0, 5) (0, 12) (0, 15) (0, 17) (0, 19) (0, 4) (1, 2) (1, 13) (1, 12) (1, 15) (1, 5) (1, 19) (1, 18) (1, 10) (1, 8) (2, 8) (2, 6) (2, 4) (2, 15) (2, 11) (2, 5) (2, 10) (2, 7) (2, 12) (2, 13) (3, 10) (3, 18) (3, 16) (3, 12) (3, 19) (3, 14) (3, 5) (3, 17) (3, 11) (3, 4) ...
```
**Answer** -> `### Yes.`

**`original`**
```
Q: The nodes are numbered from 0 to 19, and the edges are: (0, 11) (0, 16) (0, 7) (0, 5) (0, 12) (0, 15) (0, 17) (0, 19) (0, 4) (1, 2) (1, 13) (1, 12) (1, 15) (1, 5) (1, 19) (1, 18) (1, 10) (1, 8) (2, 8) (2, 6) (2, 4) (2, 15) (2, 11) (2, 5) (2, 10) (2, 7) (2, 12) (2, 13) (3, 10) (3, 18) (3, 16) (3, 12) (3, 19) (3, 14) (3, 5) (3, 17) (3, 11) (3, 4) ...
```
**Answer** -> `### Yes.`

**`perm_0`**
```
Q: The nodes are numbered from 0 to 19, and the edges are: (5,14) (1,16) (8,0) (12,11) (13,7) (5,17) (15,3) (6,10) (19,13) (7,0) (9,15) (14,16) (4,7) (10,2) (4,3) (13,18) (15,11) (4,1) (2,16) (14,6) (11,16) (19,18) (18,7) (12,10) (4,10) (17,1) (19,3) (19,7) (10,0) (12,16) (14,1) (1,3) (15,12) (6,3) (11,2) (15,7) (9,17) (6,2) (4,13) (13,1) (15,2) (9...
```
**Answer** -> `### Yes.`



### cycle

**`none`**
```
Q: The nodes are numbered from 0 to 9, and the edges are: (0, 9) (0, 6) (0, 1) (0, 5) (0, 7) (1, 9) (1, 7) (1, 4) (1, 2) (2, 7) (2, 3) (2, 8) (2, 5) (2, 9) (3, 6) (3, 9) (3, 8) (4, 8) (4, 6) (6, 8) (6, 9). [GRAPH_REPR] Is there a cycle in this graph?
```
**Answer** -> `### Yes.`

**`original`**
```
Q: The nodes are numbered from 0 to 9, and the edges are: (0, 9) (0, 6) (0, 1) (0, 5) (0, 7) (1, 9) (1, 7) (1, 4) (1, 2) (2, 7) (2, 3) (2, 8) (2, 5) (2, 9) (3, 6) (3, 9) (3, 8) (4, 8) (4, 6) (6, 8) (6, 9). [GRAPH_REPR] Is there a cycle in this graph?
```
**Answer** -> `### Yes.`

**`perm_0`**
```
Q: The nodes are numbered from 0 to 9, and the edges are: (1,0) (1,4) (6,0) (8,9) (0,4) (5,1) (8,7) (8,4) (8,3) (7,4) (5,2) (7,6) (5,9) (7,5) (1,2) (5,4) (8,0) (0,2) (6,2) (5,3) (7,9). [GRAPH_REPR] Is there a cycle in this graph?
```
**Answer** -> `### Yes.`



### shortest

**`none`**
```
Q: The nodes are numbered from 0 to 37, and the edges are: (0,16,4) (0,23,3) (0,32,10) (0,27,5) (0,12,2) (0,24,9) (0,36,7) (1,13,3) (1,5,1) (1,31,4) (1,28,1) (1,2,9) (1,22,5) (1,24,1) (1,7,10) (1,6,1) (2,25,4) (2,13,4) (2,34,2) (2,5,7) (2,18,10) (2,19,8) (2,27,1) (3,35,9) (3,13,2) (3,32,1) (3,28,6) (3,17,8) (3,37,3) (4,16,5) (4,35,4) (5,31,8) (5,34...
```
**Answer** -> `### 7.`

**`original`**
```
Q: The nodes are numbered from 0 to 37, and the edges are: (0,16,4) (0,23,3) (0,32,10) (0,27,5) (0,12,2) (0,24,9) (0,36,7) (1,13,3) (1,5,1) (1,31,4) (1,28,1) (1,2,9) (1,22,5) (1,24,1) (1,7,10) (1,6,1) (2,25,4) (2,13,4) (2,34,2) (2,5,7) (2,18,10) (2,19,8) (2,27,1) (3,35,9) (3,13,2) (3,32,1) (3,28,6) (3,17,8) (3,37,3) (4,16,5) (4,35,4) (5,31,8) (5,34...
```
**Answer** -> `### 7.`

**`perm_0`**
```
Q: The nodes are numbered from 0 to 37, and the edges are: (33,12,1) (32,16,10) (20,30,9) (27,26,3) (17,35,9) (12,2,8) (10,16,1) (33,6,7) (3,30,10) (20,37,6) (7,9,4) (32,37,7) (36,10,1) (13,0,4) (9,6,8) (7,14,1) (29,12,2) (31,0,2) (33,19,10) (20,9,10) (32,1,8) (29,17,5) (9,33,5) (23,16,6) (33,16,8) (25,37,10) (19,14,5) (37,5,5) (35,5,2) (36,7,2) (1...
```
**Answer** -> `### 7.`



### bipartite

**`none`**
```
Q: The nodes are numbered from 0 to 1, and the edges are: (0->1). [GRAPH_REPR] Is this graph bipartite?
```
**Answer** -> `### Yes.`

**`original`**
```
Q: The nodes are numbered from 0 to 1, and the edges are: (0->1). [GRAPH_REPR] Is this graph bipartite?
```
**Answer** -> `### Yes.`

**`perm_0`**
```
Q: The nodes are numbered from 0 to 1, and the edges are: (0->1). [GRAPH_REPR] Is this graph bipartite?
```
**Answer** -> `### Yes.`



### flow

**`none`**
```
Q: The nodes are numbered from 0 to 7, and the edges are: (0->7,8) (0->4,1) (0->2,9) (0->6,1) (1->7,8) (1->3,10) (1->6,5) (1->5,1) (4->5,2) (5->6,6) (6->7,2). [GRAPH_REPR] What is the maximum flow from node 4 to node 7?
```
**Answer** -> `### 2.`

**`original`**
```
Q: The nodes are numbered from 0 to 7, and the edges are: (0->7,8) (0->4,1) (0->2,9) (0->6,1) (1->7,8) (1->3,10) (1->6,5) (1->5,1) (4->5,2) (5->6,6) (6->7,2). [GRAPH_REPR] What is the maximum flow from node 4 to node 7?
```
**Answer** -> `### 2.`

**`perm_0`**
```
Q: The nodes are numbered from 0 to 7, and the edges are: (1->6,8) (7->2,6) (3->7,2) (0->5,10) (0->7,1) (1->3,1) (1->4,9) (0->6,8) (2->6,2) (0->2,5) (1->2,1). [GRAPH_REPR] What is the maximum flow from node 3 to node 6?
```
**Answer** -> `### 2.`



### topology

**`none`**
```
Q: The nodes are numbered from 0 to 16, and the edges are: (0->10) (0->1) (0->7) (0->16) (0->4) (0->3) (0->2) (0->15) (0->6) (0->13) (0->8) (0->11) (1->16) (1->4) (1->7) (1->14) (1->6) (1->5) (1->2) (1->11) (1->12) (1->15) (1->10) (1->9) (1->13) (2->15) (2->9) (2->3) (2->12) (2->14) (2->11) (2->4) (2->8) (2->6) (2->10) (2->16) (3->15) (3->14) (3->1...
```
**Answer** -> `### [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]`

**`original`**
```
Q: The nodes are numbered from 0 to 16, and the edges are: (0->10) (0->1) (0->7) (0->16) (0->4) (0->3) (0->2) (0->15) (0->6) (0->13) (0->8) (0->11) (1->16) (1->4) (1->7) (1->14) (1->6) (1->5) (1->2) (1->11) (1->12) (1->15) (1->10) (1->9) (1->13) (2->15) (2->9) (2->3) (2->12) (2->14) (2->11) (2->4) (2->8) (2->6) (2->10) (2->16) (3->15) (3->14) (3->1...
```
**Answer** -> `### [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]`

**`perm_0`**
```
Q: The nodes are numbered from 0 to 16, and the edges are: (0->16) (13->4) (2->5) (0->4) (12->10) (12->6) (11->1) (6->16) (3->11) (7->13) (7->15) (1->6) (11->13) (1->10) (7->12) (0->6) (1->12) (5->15) (0->15) (0->13) (11->4) (9->13) (8->1) (3->14) (2->15) (2->6) (5->14) (0->7) (2->4) (3->10) (2->10) (2->9) (5->7) (0->9) (8->12) (11->6) (16->13) (12...
```
**Answer** -> `### [8, 0, 2, 5, 3, 11, 7, 1, 12, 9, 10, 6, 16, 13, 4, 15, 14]`



### hamilton

**`none`**
```
Q: The nodes are numbered from 0 to 1, and the edges are: (0, 1). [GRAPH_REPR] Is there a Hamiltonian path in this graph?
```
**Answer** -> `### Yes, [0, 1] or [1, 0].`

**`original`**
```
Q: The nodes are numbered from 0 to 1, and the edges are: (0, 1). [GRAPH_REPR] Is there a Hamiltonian path in this graph?
```
**Answer** -> `### Yes, [0, 1] or [1, 0].`

**`perm_0`**
```
Q: The nodes are numbered from 0 to 1, and the edges are: (1,0). [GRAPH_REPR] Is there a Hamiltonian path in this graph?
```
**Answer** -> `### Yes, [1, 0] or [0, 1].`



### triangle

**`none`**
```
Q: The nodes are numbered from 0 to 12, weights of nodes are: [0, 9] [1, 3] [2, 6] [3, 3] [4, 6] [5, 4] [6, 7] [7, 9] [8, 4] [9, 1] [10, 7] [11, 2] [12, 9], and the edges are: (0, 10) (0, 9) (2, 6) (7, 9) (9, 10). [GRAPH_REPR] What is the maximum sum of the weights of three nodes?
```
**Answer** -> `### The maximum sum of the weights of three interconnected nodes in the graph is 17.`

**`original`**
```
Q: The nodes are numbered from 0 to 12, weights of nodes are: [0, 9] [1, 3] [2, 6] [3, 3] [4, 6] [5, 4] [6, 7] [7, 9] [8, 4] [9, 1] [10, 7] [11, 2] [12, 9], and the edges are: (0, 10) (0, 9) (2, 6) (7, 9) (9, 10). [GRAPH_REPR] What is the maximum sum of the weights of three nodes?
```
**Answer** -> `### The maximum sum of the weights of three interconnected nodes in the graph is 17.`

**`perm_0`**
```
Q: The nodes are numbered from 0 to 12, weights of nodes are: [0, 9] [1, 3] [2, 9] [3, 6] [4, 9] [5, 7] [6, 4] [7, 6] [8, 7] [9, 3] [10, 2] [11, 1] [12, 4], and the edges are: (0,11) (4,11) (0,8) (3,5) (11,8). [GRAPH_REPR] What is the maximum sum of the weights of three nodes?
```
**Answer** -> `### The maximum sum of the weights of three interconnected nodes in the graph is 17.`



### substructure

**`none`**
```
Q: The nodes of graph G are numbered from 0 to 6, and the edges are: (0->2) (0->4) (1->3) (2->5) (3->4) (4->5). The nodes of subgraph G' are numbered from a to d, and the edges are: (a->b) (a->d) (a->c) (b->d) (b->c) (c->d). [GRAPH_REPR] Is subgraph G' present within graph G as a direct substructure?
```
**Answer** -> `### No.`

**`original`**
```
Q: The nodes of graph G are numbered from 0 to 6, and the edges are: (0->2) (0->4) (1->3) (2->5) (3->4) (4->5). The nodes of subgraph G' are numbered from a to d, and the edges are: (a->b) (a->d) (a->c) (b->d) (b->c) (c->d). [GRAPH_REPR] Is subgraph G' present within graph G as a direct substructure?
```
**Answer** -> `### No.`

**`perm_0`**
```
Q: The nodes of graph G are numbered from 0 to 6, and the edges are: (0->6) (4->3) (1->4) (1->6) (6->3) (2->0). The nodes of subgraph G' are numbered from a to d, and the edges are: (a->b) (a->d) (a->c) (b->d) (b->c) (c->d). [GRAPH_REPR] Is subgraph G' present within graph G as a direct substructure?
```
**Answer** -> `### No.`



## 9. Verify `none` and `original` share the same answer

In [11]:
answer_match = 0
answer_total = 0
query_repr_match = 0

for idx, grp in aug_groups.items():
    by_aug = {s["augmentation"]: s for s in grp}
    if "none" not in by_aug or "original" not in by_aug:
        continue
    answer_total += 1
    if by_aug["none"]["answer"] == by_aug["original"]["answer"]:
        answer_match += 1
    if "[GRAPH_REPR]" in by_aug["none"]["query"] and "[GRAPH_REPR]" in by_aug["original"]["query"]:
        query_repr_match += 1

print(f"Pairs compared: {answer_total:,}")
print(f"  Answers identical (none vs original): {answer_match:,} / {answer_total:,}")
print(f"  Both have [GRAPH_REPR]:               {query_repr_match:,} / {answer_total:,}")

Pairs compared: 18,125
  Answers identical (none vs original): 18,125 / 18,125
  Both have [GRAPH_REPR]:               18,125 / 18,125


## 10. Token length distribution (SFT readiness)

In [12]:
import numpy as np

CHARS_PER_TOKEN = 4.0
prompt_template = (
    "Below is an instruction that describes a task. "
    "Write a response that appropriately completes the request step by step.\n\n"
    "### Instruction:\n{query}\n\n### Response:{answer}"
)

lengths = np.array([
    len(prompt_template.format(query=s["query"], answer=s["answer"])) / CHARS_PER_TOKEN
    for s in aug_samples
])

print("Estimated token length statistics:")
print(f"  Min:    {lengths.min():.0f}")
print(f"  Mean:   {lengths.mean():.0f}")
print(f"  Median: {np.median(lengths):.0f}")
print(f"  P95:    {np.percentile(lengths, 95):.0f}")
print(f"  P99:    {np.percentile(lengths, 99):.0f}")
print(f"  Max:    {lengths.max():.0f}")

MAX_SEQ = 2048
over = (lengths > MAX_SEQ).sum()
print(f"\nSamples exceeding {MAX_SEQ} tokens (est.): {over:,} / {len(lengths):,} ({over/len(lengths)*100:.2f}%)")

print("\nBreakdown by augmentation type:")
for aug_type in ["none", "original", "perm_0", "perm_1", "perm_2", "perm_3"]:
    idxs = [i for i, s in enumerate(aug_samples) if s["augmentation"] == aug_type]
    if not idxs:
        continue
    sub = lengths[idxs]
    over_sub = (sub > MAX_SEQ).sum()
    print(f"  {aug_type:12s}: mean={sub.mean():.0f}, max={sub.max():.0f}, >{MAX_SEQ}: {over_sub}")

Estimated token length statistics:
  Min:    128
  Mean:   401
  Median: 333
  P95:    820
  P99:    1133
  Max:    2066

Samples exceeding 2048 tokens (est.): 2 / 108,750 (0.00%)

Breakdown by augmentation type:
  none        : mean=407, max=2066, >2048: 1
  original    : mean=407, max=2066, >2048: 1
  perm_0      : mean=399, max=1987, >2048: 0
  perm_1      : mean=399, max=1984, >2048: 0
  perm_2      : mean=399, max=1991, >2048: 0
  perm_3      : mean=399, max=1986, >2048: 0
